# Metrics Summary Across Methods

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import pandas as pd
import numpy as np
from matplotlib import colors
from matplotlib.colors import LinearSegmentedColormap

from shaft_force_sensing.data import get_train_test

In [ ]:
ROOT = Path('..')
LOG_ROOT = ROOT / 'logs'
DATA_ROOT = ROOT / 'data'
OUT_DIR = ROOT / 'logs' / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

METHODS = ['transformer', 'fcn', 'lstm']
METHOD_TO_MODEL_CLS = {
    'transformer': 'LitTransformer',
    'fcn': 'LitFCN',
    'lstm': 'LitLSTM',
}

METRICS_PATTERN = re.compile(
    r'^(?P<axis>[^:]+):\s*Range=(?P<Range>[-+0-9.eE]+),\s*'
    r'RMSE=(?P<RMSE>[-+0-9.eE]+),\s*'
    r'NRMSE=(?P<NRMSE>[-+0-9.eE]+)%,\s*'
    r'R2=(?P<R2>[-+0-9.eE]+)$'
)

RUN_NAME_PATTERN = re.compile(r'^(?P<kind>base|ft|scratch)(?:_m(?P<idx>\d+))?$')
GROUP_PATTERN = re.compile(r'^C(?P<config>\d+)_(?P<contact>[FSR])_')

# Helpers

In [ ]:
def parse_metrics_file(path: Path) -> tuple[dict, pd.DataFrame]:
    lines = [ln.strip() for ln in path.read_text(encoding='utf-8').splitlines()]

    if not lines or not lines[0].startswith('Model:'):
        raise ValueError(f'Unexpected metrics format: {path}')

    header = {'Model': lines[0].split(':', 1)[1].strip()}
    header['Test data'] = lines[1].split(':', 1)[1].strip()

    dashed = [i for i, ln in enumerate(lines) if ln == '-' * 10]
    if len(dashed) < 2:
        raise ValueError(f'Cannot locate hparams section in: {path}')

    hparam_lines = lines[dashed[0] + 1 : dashed[1]]
    for ln in hparam_lines:
        if ': ' in ln:
            k, v = ln.split(': ', 1)
            header[k] = v

    rows = []
    current_group = None
    for ln in lines[dashed[1] + 1 :]:
        if not ln or ln == '-' * 10:
            continue
        if ln.startswith('Group: '):
            current_group = ln.split(': ', 1)[1]
            continue

        m = METRICS_PATTERN.match(ln)
        if m and current_group is not None:
            row = {'group': current_group, 'axis': m.group('axis')}
            row.update({
                'Range': float(m.group('Range')),
                'RMSE': float(m.group('RMSE')),
                'NRMSE': float(m.group('NRMSE')),
                'R2': float(m.group('R2')),
            })
            rows.append(row)

    return header, pd.DataFrame(rows)


def parse_run_info(run_name: str) -> tuple[str, int | None]:
    m = RUN_NAME_PATTERN.match(run_name)
    if not m:
        return 'other', None

    kind = m.group('kind')
    idx = m.group('idx')
    return kind, (int(idx) if idx is not None else None)


def extract_group_fields(group: str) -> tuple[int | None, str | None]:
    m = GROUP_PATTERN.match(group)
    if not m:
        return None, None
    return int(m.group('config')), m.group('contact')


def seen_configs_for_model(method: str, model_idx: int) -> set[int]:
    model_cls = METHOD_TO_MODEL_CLS[method]
    train_paths, _ = get_train_test(
        data_root=DATA_ROOT,
        model_cls=model_cls,
        teleop=True,
        model_idx=model_idx,
    )
    seen_configs = set()
    for p in train_paths:
        cfg, _ = extract_group_fields(p.stem)
        if cfg is not None:
            seen_configs.add(cfg)
    return seen_configs

In [ ]:
records = []

for method in METHODS:
    method_dir = LOG_ROOT / method
    if not method_dir.exists():
        continue

    for run_dir in sorted([p for p in method_dir.iterdir() if p.is_dir()]):
        run_name = run_dir.name
        run_kind, model_idx = parse_run_info(run_name)

        metrics_paths = sorted(run_dir.glob('*/metrics.txt'))
        if not metrics_paths:
            continue

        seen_config_set = set()
        if run_kind in {'ft', 'scratch'} and model_idx is not None:
            seen_config_set = seen_configs_for_model(method, model_idx)

        for metrics_path in metrics_paths:
            header, df = parse_metrics_file(metrics_path)
            if df.empty:
                continue

            for _, row in df.iterrows():
                group = row['group']
                cfg, contact = extract_group_fields(group)

                if group == 'All':
                    seen_status = 'all'
                elif run_kind in {'ft', 'scratch'} and model_idx is not None:
                    seen_status = 'seen' if cfg in seen_config_set else 'unseen'
                else:
                    seen_status = 'na'

                records.append({
                    'method': method,
                    'run_name': run_name,
                    'run_kind': run_kind,
                    'model_idx': model_idx if run_kind in {'ft', 'scratch'} else None,
                    'model_class': header.get('Model', ''),
                    'test_data': header.get('Test data', ''),
                    'group': group,
                    'config': cfg,
                    'contact': contact,
                    'seen_status': seen_status,
                    'axis': row['axis'],
                    'Range': row['Range'],
                    'RMSE': row['RMSE'],
                    'NRMSE': row['NRMSE'],
                    'R2': row['R2'],
                })

df_all = pd.DataFrame(records)
assert not df_all.empty, 'No metrics records found under logs/'

df_all.head()

In [ ]:
# 1) Base summary (Automated)
df_base = df_all[
    (df_all['run_kind'] == 'base')
    & (df_all['test_data'] == 'Automated')
    & (df_all['group'] != 'All')
].copy()

df_base = df_base.drop(columns=['model_idx'])

base_summary = df_base.sort_values(['method', 'group', 'axis'])

base_mean_by_method_axis = (
    df_base.groupby(['method', 'axis'], as_index=False)[['RMSE', 'NRMSE', 'R2']]
    .mean()
    .sort_values(['method', 'axis'])
)

base_summary.head()

In [ ]:
# 2) Fine-tune summary (Teleop) with seen/unseen and contact type
df_ft = df_all[
    (df_all['run_kind'] == 'ft')
    & (df_all['test_data'] == 'Teleop')
    & (df_all['group'] != 'All')
] .copy()

ft_summary = (
    df_ft.groupby(
        ['method', 'run_kind', 'seen_status', 'contact', 'axis'],
        as_index=False
    )[['RMSE', 'NRMSE', 'R2']]
    .mean()
    .sort_values(['method', 'run_kind', 'axis', 'seen_status', 'contact'])
)

ft_summary_no_f = ft_summary[ft_summary['contact'] != 'F'].copy()

ft_pivot_rmse = ft_summary.pivot_table(
    index=['method', 'run_kind', 'axis'],
    columns=['seen_status', 'contact'],
    values='RMSE',
)

ft_pivot_r2 = ft_summary_no_f.pivot_table(
    index=['method', 'run_kind', 'axis'],
    columns=['seen_status', 'contact'],
    values='R2',
)

ft_pivot_nrmse = ft_summary_no_f.pivot_table(
    index=['method', 'run_kind', 'axis'],
    columns=['seen_status', 'contact'],
    values='NRMSE',
)

ft_summary.head()

In [ ]:
# 3) Scratch summary (Teleop) with seen/unseen and contact type
df_scratch = df_all[
    (df_all['run_kind'] == 'scratch')
    & (df_all['test_data'] == 'Teleop')
    & (df_all['group'] != 'All')
] .copy()

scratch_summary = (
    df_scratch.groupby(
        ['method', 'run_kind', 'seen_status', 'contact', 'axis'],
        as_index=False
    )[['RMSE', 'NRMSE', 'R2']]
    .mean()
    .sort_values(['method', 'run_kind', 'axis', 'seen_status', 'contact'])
)

scratch_summary_no_f = scratch_summary[scratch_summary['contact'] != 'F'].copy()

scratch_pivot_rmse = scratch_summary.pivot_table(
    index=['method', 'run_kind', 'axis'],
    columns=['seen_status', 'contact'],
    values='RMSE',
)

scratch_pivot_r2 = scratch_summary_no_f.pivot_table(
    index=['method', 'run_kind', 'axis'],
    columns=['seen_status', 'contact'],
    values='R2',
)

scratch_pivot_nrmse = scratch_summary_no_f.pivot_table(
    index=['method', 'run_kind', 'axis'],
    columns=['seen_status', 'contact'],
    values='NRMSE',
)

scratch_summary.head()

# Tables generation

Table IV: Automated Evaluations

In [ ]:
# Automated benchmark table (mean only) for Transformer, LSTM, and FCN.
# Use raw 'Group: All' records directly from metrics output.

auto_df = df_all[
    (df_all['test_data'] == 'Automated')
    & (df_all['run_kind'] == 'base')
    & (df_all['group'] == 'All')
    & (df_all['axis'].isin(['F_x', 'F_y', 'F_z']))
].copy()

axis_order = ['F_x', 'F_y', 'F_z']
axis_display = {'F_x': '$F_x$', 'F_y': '$F_y$', 'F_z': '$F_z$'}

# If multiple runs exist for a method, keep robust behavior by averaging those 'All' rows.
summary = (
    auto_df.groupby(['axis', 'method'], as_index=False)[['Range', 'RMSE', 'NRMSE', 'R2']]
    .mean()
)

def get_metric(axis: str, method: str, metric: str) -> float:
    hit = summary[(summary['axis'] == axis) & (summary['method'] == method)]
    if hit.empty:
        return float('nan')
    return float(hit.iloc[0][metric])

# Find best (minimum) NRMSE per axis across all three methods
best_nrmse_per_axis = {}
for axis in axis_order:
    nrmse_values = []
    for method in ['transformer', 'lstm', 'fcn']:
        nrmse = get_metric(axis, method, 'NRMSE')
        if not pd.isna(nrmse):
            nrmse_values.append(nrmse)
    if nrmse_values:
        best_nrmse_per_axis[axis] = min(nrmse_values)
    else:
        best_nrmse_per_axis[axis] = float('inf')

lines = []
lines.append(r'\begin{table*}[t]')
lines.append(r'\centering')
lines.append(r'\caption{System performance comparison on the Automated test set (best NRMSE per axis in bold)}')
lines.append(r'\label{tab:automated_results}')
lines.append(r'\setlength{\tabcolsep}{4pt}')
lines.append(r'\renewcommand{\arraystretch}{1.15}')
lines.append(r'\begin{tabular}{c c ccc ccc ccc}')
lines.append(r'\toprule')
lines.append(r'\multirow{2}{*}{Axis} & \multirow{2}{*}{Range (\si{\newton})}')
lines.append(r'& \multicolumn{3}{c}{Transformer}')
lines.append(r'& \multicolumn{3}{c}{LSTM~\cite{yang2025effectiveness}}')
lines.append(r'& \multicolumn{3}{c}{FCN} \\')
lines.append(r'\cmidrule(lr){3-5}\cmidrule(lr){6-8}\cmidrule(lr){9-11}')
lines.append(r'& & RMSE (\si{\newton}) & NRMSE (\%) & $\mathbf{R^2}$ & RMSE (\si{\newton}) & NRMSE (\%) & $\mathbf{R^2}$ & RMSE (\si{\newton}) & NRMSE (\%) & $\mathbf{R^2}$ \\')
lines.append(r'\midrule')

for axis in axis_order:
    range_val = get_metric(axis, 'transformer', 'Range')

    t_rmse = get_metric(axis, 'transformer', 'RMSE')
    t_nrmse = get_metric(axis, 'transformer', 'NRMSE')
    t_r2 = get_metric(axis, 'transformer', 'R2')

    l_rmse = get_metric(axis, 'lstm', 'RMSE')
    l_nrmse = get_metric(axis, 'lstm', 'NRMSE')
    l_r2 = get_metric(axis, 'lstm', 'R2')

    f_rmse = get_metric(axis, 'fcn', 'RMSE')
    f_nrmse = get_metric(axis, 'fcn', 'NRMSE')
    f_r2 = get_metric(axis, 'fcn', 'R2')

    # Format NRMSE values, bolding the best one for this axis
    t_nrmse_str = f"$\\mathbf{{{t_nrmse:.2f}}}$" if abs(t_nrmse - best_nrmse_per_axis[axis]) < 1e-6 else f"${t_nrmse:.2f}$"
    l_nrmse_str = f"$\\mathbf{{{l_nrmse:.2f}}}$" if abs(l_nrmse - best_nrmse_per_axis[axis]) < 1e-6 else f"${l_nrmse:.2f}$"
    f_nrmse_str = f"$\\mathbf{{{f_nrmse:.2f}}}$" if abs(f_nrmse - best_nrmse_per_axis[axis]) < 1e-6 else f"${f_nrmse:.2f}$"

    lines.append(
        f"{axis_display[axis]} & ${range_val:.2f}$ "
        f"& ${t_rmse:.2f}$ & {t_nrmse_str} & ${t_r2:.2f}$ "
        f"& ${l_rmse:.2f}$ & {l_nrmse_str} & ${l_r2:.2f}$ "
        f"& ${f_rmse:.2f}$ & {f_nrmse_str} & ${f_r2:.2f}$ \\\\"
    )

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table*}')

auto_latex_path = OUT_DIR / 'automated_benchmark.tex'
auto_latex_path.write_text('\n'.join(lines), encoding='utf-8')

print(f'Saved Automated benchmark LaTeX: {auto_latex_path}')

Table V: Teleop Evaluations

In [ ]:
# Build Teleop transfer learning results table with Seen/Unseen split.
axis_order = ['F_x', 'F_y', 'F_z']
axis_display = {'F_x': '$F_x$', 'F_y': '$F_y$', 'F_z': '$F_z$'}
contact_order = ['Rigid', 'Soft']
seen_order = ['seen', 'unseen']
seen_display = {'seen': 'Seen', 'unseen': 'Unseen'}

metric_cols = ['Range', 'RMSE', 'NRMSE', 'R2']

model_specs = [
    ('Transformer (Ours)', 'transformer', 'ft'),
    (r'LSTM~\cite{yang2025effectiveness}', 'lstm', 'ft'),
]

def build_summary_table(df_all, method: str, run_kind: str) -> pd.DataFrame:
    """Build summary table for a specific method/run_kind combination."""
    block = df_all[
        (df_all['method'] == method)
        & (df_all['run_kind'] == run_kind)
        & (df_all['test_data'] == 'Teleop')
        & (df_all['group'] != 'All')
        & (df_all['axis'].isin(axis_order))
        & (df_all['contact'].isin(['R', 'S']))
        & (df_all['seen_status'].isin(seen_order))
    ].copy()

    block['contact_display'] = block['contact'].map({'R': 'Rigid', 'S': 'Soft'})
    block['seen_display'] = block['seen_status'].map(seen_display)

    # Average within each model/run first, then average across runs.
    per_run = block.groupby(
        ['model_idx', 'contact_display', 'seen_display', 'axis'],
        as_index=False,
    )[metric_cols].mean()
    summary = per_run.groupby(
        ['contact_display', 'seen_display', 'axis'],
        as_index=False,
    )[metric_cols].agg(['mean', 'std'])

    return summary

# Collect summary data for all models.
all_summaries = {}
for model_label, method, run_kind in model_specs:
    all_summaries[(method, run_kind)] = build_summary_table(df_all, method, run_kind)


def get_value_with_std(df_summary: pd.DataFrame, contact: str, seen_status: str, axis: str, metric: str) -> tuple[float, float]:
    """Extract mean and std for a metric."""
    hit = df_summary[
        (df_summary[('contact_display', '')] == contact)
        & (df_summary[('seen_display', '')] == seen_display[seen_status])
        & (df_summary[('axis', '')] == axis)
    ]
    if hit.empty:
        return float('nan'), float('nan')
    mean_val = float(hit.iloc[0][(metric, 'mean')])
    std_val = float(hit.iloc[0][(metric, 'std')])
    return mean_val, std_val

# Build row data for each (contact, seen_status, axis) pair.
rows_data = {}
for contact in contact_order:
    for seen_status in seen_order:
        for axis in axis_order:
            rows_data[(contact, seen_status, axis)] = {}
            for (method, run_kind), df_summary in all_summaries.items():
                rows_data[(contact, seen_status, axis)][(method, run_kind)] = {}
                for metric in metric_cols:
                    mean_val, std_val = get_value_with_std(df_summary, contact, seen_status, axis, metric)
                    rows_data[(contact, seen_status, axis)][(method, run_kind)][metric] = (mean_val, std_val)

# Collect all metric values for color normalization.
all_rmse_means = []
all_nrmse_means = []
all_r2_means = []

for contact in contact_order:
    for seen_status in seen_order:
        for axis in axis_order:
            for (method, run_kind) in all_summaries.keys():
                rmse_mean, _ = rows_data[(contact, seen_status, axis)][(method, run_kind)]['RMSE']
                nrmse_mean, _ = rows_data[(contact, seen_status, axis)][(method, run_kind)]['NRMSE']
                r2_mean, _ = rows_data[(contact, seen_status, axis)][(method, run_kind)]['R2']

                if not pd.isna(rmse_mean):
                    all_rmse_means.append(rmse_mean)
                if not pd.isna(nrmse_mean):
                    all_nrmse_means.append(nrmse_mean)
                if not pd.isna(r2_mean):
                    all_r2_means.append(r2_mean)

# Create colormaps for each metric.
if all_rmse_means:
    teleop_rmse_norm = colors.Normalize(
        vmin=float(min(all_rmse_means)),
        vmax=float(max(all_rmse_means)),
    )
else:
    teleop_rmse_norm = colors.Normalize(vmin=0.0, vmax=1.0)

teleop_rmse_cmap = LinearSegmentedColormap.from_list(
    'teleop_rmse_scale', ['#63BE7B', '#FFEB84', '#F8696B'], N=256
)

if all_nrmse_means:
    teleop_nrmse_norm = colors.Normalize(
        vmin=float(min(all_nrmse_means)),
        vmax=float(max(all_nrmse_means)),
    )
else:
    teleop_nrmse_norm = colors.Normalize(vmin=0.0, vmax=1.0)

teleop_nrmse_cmap = LinearSegmentedColormap.from_list(
    'teleop_nrmse_scale', ['#63BE7B', '#FFEB84', '#F8696B'], N=256
)

if all_r2_means:
    teleop_r2_norm = colors.Normalize(
        vmin=float(min(all_r2_means)),
        vmax=float(max(all_r2_means)),
    )
else:
    teleop_r2_norm = colors.Normalize(vmin=0.0, vmax=1.0)

teleop_r2_cmap = LinearSegmentedColormap.from_list(
    'teleop_r2_scale', ['#F8696B', '#FFEB84', '#63BE7B'], N=256
)


def get_teleop_cell_color_hex(value: float, cmap, norm) -> str:
    r, g, b, _ = cmap(norm(value))
    return f'{int(r * 255):02X}{int(g * 255):02X}{int(b * 255):02X}'


def format_teleop_colored_cell(mean_val: float, std_val: float, cmap, norm) -> str:
    """Format cell with background color for mean ± std."""
    if pd.isna(mean_val):
        return '--'
    hex_color = get_teleop_cell_color_hex(mean_val, cmap, norm)
    return rf'\teleopcolorcell{{{hex_color}}}{{{mean_val:.2f}}}{{{std_val:.2f}}}'

# Generate LaTeX table.
lines = []
lines.append(r'\begin{table*}[t]')
lines.append(r'\centering')
lines.append(r'\caption{\newstuff{Finetune performance comparison across teleoperation dataset \\ (mean $\pm$ std; color intensity: green=better, red=worse)}}')
lines.append(r'\label{tab:transfer_learning_results}')
lines.append(r'\begin{tabular}{c c c c ccc ccc}')
lines.append(r'\toprule')
lines.append(r'\multirow{2}{*}{Condition} & \multirow{2}{*}{Config} & \multirow{2}{*}{Axis} & \multirow{2}{*}{Range (\si{\newton})}')
lines.append(r'& \multicolumn{3}{c}{Transformer (Ours)}')
lines.append(r'& \multicolumn{3}{c}{LSTM~\cite{yang2025effectiveness}} \\')
lines.append(r'\cmidrule(lr){5-7}\cmidrule(lr){8-10}')
lines.append(r'& & & & RMSE (\si{\newton}) & NRMSE (\%) & $\mathbf{R^2}$ & RMSE (\si{\newton}) & NRMSE (\%) & $\mathbf{R^2}$ \\')
lines.append(r'\midrule')

for contact_idx, contact in enumerate(contact_order):
    for seen_idx, seen_status in enumerate(seen_order):
        for axis_idx, axis in enumerate(axis_order):
            contact_cell = rf'\multirow{{6}}{{*}}{{{contact}}}' if seen_idx == 0 and axis_idx == 0 else ''
            seen_cell = rf'\multirow{{3}}{{*}}{{{seen_display[seen_status]}}}' if axis_idx == 0 else ''
            first_key = (model_specs[0][1], model_specs[0][2])
            range_mean, range_std = rows_data[(contact, seen_status, axis)][first_key]['Range']
            range_str = f'${range_mean:.2f} \\pm {range_std:.2f}$' if not pd.isna(range_mean) else '--'
            row_cells = [contact_cell, seen_cell, axis_display[axis], range_str]
            for model_label, method, run_kind in model_specs:
                rmse_mean, rmse_std = rows_data[(contact, seen_status, axis)][(method, run_kind)]['RMSE']
                nrmse_mean, nrmse_std = rows_data[(contact, seen_status, axis)][(method, run_kind)]['NRMSE']
                r2_mean, r2_std = rows_data[(contact, seen_status, axis)][(method, run_kind)]['R2']
                row_cells.append(format_teleop_colored_cell(rmse_mean, rmse_std, teleop_rmse_cmap, teleop_rmse_norm))
                row_cells.append(format_teleop_colored_cell(nrmse_mean, nrmse_std, teleop_nrmse_cmap, teleop_nrmse_norm))
                row_cells.append(format_teleop_colored_cell(r2_mean, r2_std, teleop_r2_cmap, teleop_r2_norm))
            lines.append(' & '.join(row_cells) + r' \\')
        if seen_idx == 0:
            lines.append(r'\cmidrule(lr){2-10}')
    if contact_idx < len(contact_order) - 1:
        lines.append(r'\midrule')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table*}')

teleop_benchmark_path = OUT_DIR / 'teleop_transfer_learning_results.tex'
teleop_benchmark_path.write_text('\n'.join(lines), encoding='utf-8')

print(f'Saved Teleop transfer learning LaTeX table: {teleop_benchmark_path}')

Table VI: NRMSE Color Map

In [ ]:
# Generate Teleop NRMSE table using the updated LaTeX template style.
seen_unseen_spec = [
    (r'Transformer\\(Scratch)', 'transformer', 'scratch'),
    (r'Transformer\\(Finetune)', 'transformer', 'ft'),
    (r'LSTM~\\cite{yang2025effectiveness}\\(Finetune)', 'lstm', 'ft'),
    # (r'FCN\\(Finetune)', 'fcn', 'ft'),
]
seen_unseen_axis_order = ['F_x', 'F_y', 'F_z']
seen_unseen_contact_order = [('seen', 'R'), ('seen', 'S'), ('unseen', 'R'), ('unseen', 'S')]
seen_unseen_display = {'F_x': '$F_x$', 'F_y': '$F_y$', 'F_z': '$F_z$'}

def build_seen_unseen_block(summary_df: pd.DataFrame, method: str, run_kind: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    block = summary_df[
        (summary_df['method'] == method)
        & (summary_df['run_kind'] == run_kind)
        & (summary_df['contact'].isin(['R', 'S']))
        & (summary_df['axis'].isin(seen_unseen_axis_order))
        & (summary_df['group'] != 'All')
    ].copy()

    grouped = (
        block.groupby(['axis', 'seen_status', 'contact'], as_index=False)['NRMSE']
        .agg(['mean', 'std'])
        .reset_index()
    )
    mean_table = grouped.pivot(index='axis', columns=['seen_status', 'contact'], values='mean')
    std_table = grouped.pivot(index='axis', columns=['seen_status', 'contact'], values='std')
    mean_table = mean_table.reindex(index=seen_unseen_axis_order)
    std_table = std_table.reindex(index=seen_unseen_axis_order)
    mean_table = mean_table.reindex(columns=pd.MultiIndex.from_tuples(seen_unseen_contact_order))
    std_table = std_table.reindex(columns=pd.MultiIndex.from_tuples(seen_unseen_contact_order))
    return mean_table, std_table

seen_unseen_blocks = []
all_seen_unseen_values = []
for model_label, method, run_kind in seen_unseen_spec:
    mean_table, std_table = build_seen_unseen_block(df_all, method, run_kind)
    seen_unseen_blocks.append((model_label, mean_table, std_table))
    all_seen_unseen_values.extend(
        pd.Series(mean_table.to_numpy().ravel()).dropna().astype(float).tolist()
    )

if all_seen_unseen_values:
    seen_norm = colors.Normalize(
        vmin=float(min(all_seen_unseen_values)),
        vmax=float(max(all_seen_unseen_values)),
    )
else:
    seen_norm = colors.Normalize(vmin=0.0, vmax=1.0)

seen_cmap = LinearSegmentedColormap.from_list(
    'seen_scale', ['#63BE7B', '#FFEB84', '#F8696B'], N=256
)

def seen_cell_color_hex(value: float) -> str:
    r, g, b, _ = seen_cmap(seen_norm(value))
    return f'{int(r * 255):02X}{int(g * 255):02X}{int(b * 255):02X}'

def safe_float(value: float | int | None, default: float = 0.0) -> float:
    if value is None or pd.isna(value):
        return default
    return float(value)

lines = []
lines.append(r'\begin{table}[t]')
lines.append(r'\centering')
lines.append(r'\caption{Comparsion of \gls{nrmse} (\%)\\(mean $\pm$ std over all cross-validation models)}')
lines.append(r'\label{tab:teleop_nrmse}')
lines.append(r'\resizebox{\columnwidth}{!}{%')
lines.append(r'\begin{tabular}{llcccc}')
lines.append(r'\toprule')
lines.append(r'\multirow{2}{*}{\makecell[c]{Method}} & \multirow{2}{*}{\makecell[c]{Axis}} & \multicolumn{2}{c}{Seen} & \multicolumn{2}{c}{Unseen} \\')
lines.append(r'\cmidrule(lr){3-4} \cmidrule(lr){5-6}')
lines.append(r'& & Rigid & Soft & Rigid & Soft \\')
lines.append(r'\midrule')

for model_idx, (model_label, mean_table, std_table) in enumerate(seen_unseen_blocks):
    row_span = len(seen_unseen_axis_order)
    for axis_idx, axis_name in enumerate(seen_unseen_axis_order):
        method_cell = rf'\multirow{{{row_span}}}{{*}}{{\makecell[c]{{{model_label}}}}}' if axis_idx == 0 else ''

        row_cells = [method_cell, seen_unseen_display[axis_name]]
        for status, contact in seen_unseen_contact_order:
            mean_value = safe_float(mean_table.loc[axis_name, (status, contact)], default=float('nan'))
            std_value = safe_float(std_table.loc[axis_name, (status, contact)], default=0.0)

            if pd.isna(mean_value):
                row_cells.append(r'--')
                continue

            hex_color = seen_cell_color_hex(mean_value)
            row_cells.append(rf'\scorecell{{{hex_color}}}{{{mean_value:.2f}}}{{{std_value:.2f}}}')

        lines.append(' & '.join(row_cells) + r' \\')

    if model_idx < len(seen_unseen_blocks) - 1:
        lines.append(r'\midrule')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}%')
lines.append(r'}')
lines.append(r'\end{table}')

seen_template_path = OUT_DIR / 'nrmse_table_seen_unseen_template.tex'
seen_template_path.write_text('\n'.join(lines), encoding='utf-8')

print(f'Saved template-style LaTeX table: {seen_template_path}')

Save as xlsx files

In [ ]:
automated_xlsx = OUT_DIR / 'automated.xlsx'
teleop_ft_xlsx = OUT_DIR / 'teleop finetune.xlsx'
teleop_scratch_xlsx = OUT_DIR / 'teleop scratch.xlsx'

with pd.ExcelWriter(automated_xlsx, engine='openpyxl') as writer:
    base_mean_by_method_axis.to_excel(writer, sheet_name='base_mean_axis', index=False)
    base_summary.to_excel(writer, sheet_name='base_raw', index=False)

with pd.ExcelWriter(teleop_ft_xlsx, engine='openpyxl') as writer:
    ft_pivot_rmse.to_excel(writer, sheet_name='pivot_rmse')
    ft_pivot_r2.to_excel(writer, sheet_name='pivot_r2')
    ft_pivot_nrmse.to_excel(writer, sheet_name='pivot_nrmse')
    df_ft.to_excel(writer, sheet_name='teleop_raw', index=False)

with pd.ExcelWriter(teleop_scratch_xlsx, engine='openpyxl') as writer:
    scratch_pivot_rmse.to_excel(writer, sheet_name='pivot_rmse')
    scratch_pivot_r2.to_excel(writer, sheet_name='pivot_r2')
    scratch_pivot_nrmse.to_excel(writer, sheet_name='pivot_nrmse')
    df_scratch.to_excel(writer, sheet_name='teleop_raw', index=False)

print(f'Wrote: {automated_xlsx}')
print(f'Wrote: {teleop_ft_xlsx}')
print(f'Wrote: {teleop_scratch_xlsx}')